# OSASUD: пересмотренный эксперимент для статьи

**30 пациентов, по 30 минут на пациента; nested patient-group CV; три модели.**
Это новая редакция загруженного ноутбука. Результаты получены реальным запуском,
не перенесены из короткого примера. Исходные значения не изменены.
Встроены признаки только обрезанных записей, таблица контроля качества, OOF-прогнозы, код
выделения признаков и по **30 минут настоящих исходных сигналов пациентов 3,5,6**.
Сам полный Pickle 3,28 ГБ не встроен. Ни одна выполняемая ячейка этого блокнота не открывает полный Pickle: только небольшие NPZ.
Источник: https://pmc.ncbi.nlm.nih.gov/articles/PMC9018698/ ; DOI 10.6084/m9.figshare.19214289.v1.
Проверенный MD5: b9ed560f4abf6100ce02cfc7cb0c121f. Лицензия CC0.

Запускайте ячейки сверху вниз. По умолчанию показываются сохранённые результаты
проверенного запуска. Для независимого повторного обучения установите RECOMPUTE=True.
Зависимости перечислены в requirements.txt. При отсутствии библиотек в Colab:
`%pip install numpy pandas scipy matplotlib scikit-learn`.


**Это обрезанная версия.** Внутри 30 небольших patient_N.npz. Большого файла нет и код его не читает. Для исходников отдельно доступен OSASUD_30min_raw.zip; для запуска этого блокнота скачивать его отдельно не нужно. Все метрики пересчитаны только на обрезанной выборке.


In [1]:
# @title Открыть только обрезанные записи и пересчитать признаки (без большого Pickle)
import base64,io,zipfile,hashlib,json,sys
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display,Image
work=Path.cwd()/'OSASUD_cropped30_work'
work.mkdir(exist_ok=True)
packed=base64.b64decode('...')
assert hashlib.sha256(packed).hexdigest()=='44f280487712c455c98004a6488177267ffcb1e6b6e956057769447f8222f00b'
with zipfile.ZipFile(io.BytesIO(packed)) as z:
    for name in z.namelist():
        if Path(name).name!=name:raise ValueError('Unexpected archive path')
        (work/name).write_bytes(z.read(name))
for name,digest in json.loads((work/'checksums.json').read_text()).items():
    assert hashlib.sha256((work/name).read_bytes()).hexdigest()==digest,name
sys.path.insert(0,str(work))
for name in ['signal_features','raw_windows','evaluate','plots']:sys.modules.pop(name,None)
from raw_windows import rebuild
manifest=json.loads((work/'manifest.json').read_text())
frame,quality=rebuild(work)
print('Встроены только 30 минут каждого из 30 пациентов: 15 часов, без полного Pickle.')
print('Полных минут до QC:',len(quality),'после QC:',len(frame),'для бинарной оценки:',int(frame.y.notna().sum()))
display(frame.head(3))


Встроены только 30 минут каждого из 30 пациентов: 15 часов, без полного Pickle.
Полных минут до QC: 900 после QC: 843 для бинарной оценки: 654
  patient  segment  source_row  ... flow_P90_P10  flow_derivative_SD  flow_entropy
0       1        1        2100  ...     2.739013            2.556907      0.647274
1       1        1        2160  ...     2.450943            2.026721      0.411358
2       1        1        2220  ...     2.315242            2.090063      0.558925

[3 rows x 25 columns]


## 1. Постановка, отбор и контроль качества
Одна строка оригинала — одна секунда; signal_pleth 80 Гц, PSG_Flow 20 Гц.
Анализируются по 30 минут 30 пациентов: всего 900 неперекрывающихся окон по 60 с до QC. Для каждого пациента выбран первый по времени 30-минутный блок с >=15 минутами, принятыми по QC. Если такого блока нет, берётся первый. Правило не использует event или успех модели. Такая выборка смещена по качеству и времени ночи и не представляет всю ночь.
Окна не пересекают разрывы времени. Окна с >=10 секунд APNEA — положительные,
с целой минутой NONE — отрицательные. Гипопноэ, неизвестные метки и 1–9 секунд
APNEA исключены из бинарной задачи; они остаются на исходных графиках.
Это не AHI и не детектор точных границ эпизодов.

Общая для моделей complete-case выборка: отвергаются окна с NaN/Inf в ФПГ или потоке,
постоянным участком ФПГ >=2 с, плоским ФПГ, недостаточными пиками или >20% неправдоподобных
PPI. Критерии заданы заранее, без использования event. Отдельно публикуются причины,
покрытие и вариация по пациентам. Исключение плохих окон может создавать selection bias;
качество на отвергнутых данных неизвестно. Эти простые QC-признаки не гарантируют отсутствие артефактов.
HR(bpm) из ЭКГ не используется как вход или критерий отбора модели.


In [2]:
display(quality.groupby(['quality','label_eligible']).size().rename('windows').reset_index())
display(pd.read_csv(work/'coverage_by_patient.csv').head(20))


                 quality  label_eligible  windows
0  PPG constant run >=2s            True        2
1               accepted           False      189
2               accepted            True      654
3                missing           False        7
4                missing            True       48
    patient                quality  label_eligible  windows
0         1               accepted           False        4
1         1               accepted            True       21
2         1                missing            True        5
3        10               accepted           False       11
4        10               accepted            True       19
5        11               accepted            True       30
6        12               accepted            True       30
7        13               accepted           False        7
8        13               accepted            True       22
9        13                missing            True        1
10       14  PPG constant run >=2s      

## 2. Предобработка и признаки — без утечки
Удалена RMS-нормализация всей записи и вычисление общего baseline. Нет глобальной
интерполяции и передискретизации. На отдельной копии каждого окна применяются фиксированные
фильтры: signal_pleth 0,5–8 Гц, PSG_Flow 0,05–2 Гц, Баттерворт 3-го порядка, zero-phase.
Это офлайн-обработка: будущие отсчёты внутри окна используются.

Признаки пересчитываются непосредственно из встроенных небольших NPZ при запуске. Фиксированные локальные RMS/SD/спектральные признаки не оценивают параметры по выборке,
поэтому их можно кешировать до CV: каждое окно независимо от остальных, event не используется.
Импутация признаков, standardization, class weighting и выбор C обучаются только в train
каждого внутреннего/внешнего фолда. Измерения с пропусками не интерполируются.

PPG: PR=60/median(PPI), SD PPI, RMSSD PPI, prominence median/CV, SD ФПГ, спектральная энтропия.
Пики: расстояние >=28 отсчётов, prominence >=0,3 SD; допустимый PPI 0,35–2 с.
Flow: RMS, SD, P90−P10, SD производной, спектральная энтропия.
Tp, App, ST, DT, Smax и E(Tp) в этой версии **не рассчитываются и не заявляются**.
Ссылка на непроверенное воспроизведение сторонней нейросети удалена.


In [3]:
import inspect
from signal_features import features
print(inspect.getsource(features))
print('PPG:',manifest['PPG_features'])
print('Flow:',manifest['flow_features'])


def features(x,f):
    quality=dict(ppg_missing=float(np.mean(~np.isfinite(x))),flow_missing=float(np.mean(~np.isfinite(f))),
        ppg_constant_run_s=longest(np.r_[False,np.diff(x)==0])/80)
    if not np.isfinite(x).all() or not np.isfinite(f).all():return {},quality,'missing'
    if quality['ppg_constant_run_s']>=2:return {},quality,'PPG constant run >=2s'
    y=signal.sosfiltfilt(SOS,x)
    if np.std(y)<1e-10:return {},quality,'flat PPG'
    peaks,props=signal.find_peaks(y,distance=28,prominence=.3*np.std(y))
    ppi=np.diff(peaks)/80;ok=(ppi>=.35)&(ppi<=2.)
    quality['invalid_PPI_fraction']=float(1-ok.mean()) if len(ok) else 1.
    if ok.sum()<15 or quality['invalid_PPI_fraction']>.2:return {},quality,'unreliable peaks'
    d=np.diff(ppi)[ok[:-1]&ok[1:]]
    if len(d)<2:return {},quality,'few intervals'
    amp=props['prominences'];good=ppi[ok]
    values=[60/np.median(good),np.std(good),np.sqrt(np.mean(d*d)),np.median(amp),np.std(amp)/np.mean(amp),np.std(y),entropy(y,80,.5,8)]

## 3. Вложенная проверка по пациентам, калибровка и порог
Сравниваются PPG, flow_only и PPG_plus_flow на одинаковых окнах и внешних фолдах.
Внешний GroupKFold(5): пациенты целиком отложены, каждый пациент в тесте один раз.
Это НЕ leave-one-patient-out; каждый внешний тест содержит группу пациентов.
Внутри train — GroupKFold(3). C∈{0.1,1,10} выбирается по внутреннему average precision.
По внутренним OOF-оценкам выбранной модели обучается sigmoid/Platt-калибратор без class weighting.
Порог максимизирует внутренний F1 после калибровки. Внешние event и оценки не участвуют
ни в выборе C/порога, ни в обучении калибратора. Порог может различаться между фолдами.
Выбор порога и калибровка используют одну внутреннюю OOF-выборку; честная итоговая оценка
обеспечивается независимым внешним фолдом.

ROC-AUC, PR-AUC (именно average precision, не трапецеидальная площадь), Brier,
precision/recall/F1 и balanced accuracy считаются по внешним OOF-вероятностям.
Есть показатели каждого пациента и macro-усреднение, чтобы длинные записи не скрывали различия.
95% CI: 500 bootstrap-выборок целых пациентов, seed=42. Это условные CI по готовым OOF-прогнозам:
модели в bootstrap не переобучаются; полная неопределённость выбора модели ими не охватывается.
Не заявляем статистическое превосходство только по этим CI.


In [4]:
RECOMPUTE=False  # True: повторить вложенный CV и 500 bootstrap-выборок
from evaluate import evaluate
if RECOMPUTE:
    evaluate(frame,manifest['PPG_features'],manifest['flow_features'],out=work,boots=500)
metrics=pd.read_csv(work/'metrics.csv')
ci=pd.read_csv(work/'cluster_bootstrap_CI.csv')
display(metrics)
display(ci)
display(pd.read_csv(work/'fold_parameters.csv'))
display(pd.read_csv(work/'macro_patient_metrics.csv'))


           model  windows  patients  ...  precision        F1  balanced_accuracy
0            PPG      654        30  ...   0.010989  0.012270           0.429625
1  PPG_plus_flow      654        30  ...   0.225806  0.285714           0.611970
2      flow_only      654        30  ...   0.226667  0.231293           0.568227

[3 rows x 13 columns]
            model             metric  ...     upper  valid_bootstraps
0             PPG            ROC_AUC  ...  0.469632               500
1             PPG             PR_AUC  ...  0.179884               500
2             PPG              Brier  ...  0.178096               500
3             PPG           accuracy  ...  0.851795               500
4             PPG        sensitivity  ...  0.066667               500
5             PPG        specificity  ...  0.935554               500
6             PPG          precision  ...  0.043675               500
7             PPG                 F1  ...  0.041149               500
8             PPG  bala

## 4. ROC, PR и внешняя калибровка
Калибровочная кривая построена только по внешним OOF-прогнозам: идеальная диагональ
показывает совпадение вероятности с частотой события. Brier указан в таблице.
Калибровочные интервалы по квантилям, до 10 групп. PR-baseline — доля положительных окон.


In [5]:
from plots import plots
plots(work)
display(Image(filename=str(work/'figures/ROC_PR_calibration.png')))
display(pd.read_csv(work/'metrics_by_fold.csv'))


<IPython.core.display.Image object>
            model  fold   ROC_AUC  ...  precision        F1  balanced_accuracy
0             PPG     1  0.324925  ...   0.000000  0.000000           0.495495
1             PPG     2  0.298920  ...   0.000000  0.000000           0.361345
2             PPG     3  0.377953  ...   0.017544  0.033333           0.446194
3             PPG     4  0.274336  ...        NaN  0.000000           0.500000
4             PPG     5  0.669643  ...        NaN  0.000000           0.500000
5   PPG_plus_flow     1  0.687688  ...   0.194444  0.274510           0.602703
6   PPG_plus_flow     2  0.865546  ...   0.470588  0.516129           0.747899
7   PPG_plus_flow     3  0.724409  ...   0.055556  0.102564           0.699475
8   PPG_plus_flow     4  0.651770  ...   0.269231  0.304348           0.590929
9   PPG_plus_flow     5  0.795982  ...   0.444444  0.275862           0.577679
10      flow_only     1  0.749550  ...   0.125000  0.086957           0.501802
11      flow_onl

## 5. Проверка пульса по доступному HR(bpm)
Сравниваем минутный PR из signal_pleth с медианным HR(bpm) монитора по ЭКГ,
включая все принятые по QC окна, даже если они не подходят бинарной задаче.
Показаны MAE, RMSE, bias и Bland–Altman. Это проверка согласия частот, **не ручная
валидация каждого пика**. Пульсовой дефицит, аритмии и ошибки монитора могут создавать расхождения.
LoA описательные: повторные окна одного пациента зависимы. Для окончательной валидации пиков
нужна ручная выборочная разметка или сопоставление с валидированными ECG R-peaks.


In [6]:
display(json.loads((work/'pulse_validation.json').read_text()))
display(Image(filename=str(work/'figures/pulse_validation.png')))


{'windows': 843, 'patients': 30, 'MAE': 0.8976387773452764, 'RMSE': 1.9281279642825375, 'bias': 0.0303299309595989, 'limits_of_agreement': [-3.7505764906584464, 3.811236352577644], 'caveat': 'Rate agreement only, not validation of individual peak positions; ECG heart rate and peripheral pulse can differ.'}
<IPython.core.display.Image object>


## 6. Длинные исходные фрагменты и вероятности моделей
По 30 минут пациентов 3,5,6; выбор Для иллюстраций выбраны первые три ID с >=60 секундами APNEA среди уже зафиксированных обрезанных записей. Только выбор рисунков использует event; состав обучающей/тестовой выборки и пороги от этого не зависят.
Исходники без изменения. Синий — expert event APNEA, жёлтый — HYPOPNEA.
Вероятности для этих пациентов — только из внешних тестовых фолдов.
Пунктир — порог из внутреннего CV этого фолда, красный — положительное минутное окно;
серый — окно не вошло в общую оценку. На графиках время в минутах; увеличение пиков — в секундах.

**Постобработка:** закрытия разрывов нет (max_gap=0); короткие предсказания не удаляются.
Каждое положительное окно равно 60 с. Соседние окна могут визуально соприкасаться;
через исключённые окна интервалы не объединяются. Event-level метрики не выдаются:
подсветка минут не определяет точные начало/конец клинического события.


In [7]:
for patient in ['3','5','6']:
    print('patient',patient,manifest['examples'][patient])
    display(Image(filename=str(work/f'figures/patient_{patient}_30min.png')))
    if (work/f'figures/patient_{patient}_peaks.png').exists():
        display(Image(filename=str(work/f'figures/patient_{patient}_peaks.png')))


patient 3 {'start_source_row': 644357, 'end_source_row': 646156, 'start_time': '2019-08-05 21:59:39', 'duration_seconds': 1800, 'accepted_minutes_in_clip': 30, 'sha256': '1889065fc2d67f91a8e750a112e9eddd1b36a33f7e0c794c63301327709b2d36'}
<IPython.core.display.Image object>
<IPython.core.display.Image object>
patient 5 {'start_source_row': 814305, 'end_source_row': 816104, 'start_time': '2019-08-08 21:54:53', 'duration_seconds': 1800, 'accepted_minutes_in_clip': 30, 'sha256': 'eb4089bd1181e48215b89317dab5d77b3f646f858c652e5e010124bcaab606a6'}
<IPython.core.display.Image object>
<IPython.core.display.Image object>
patient 6 {'start_source_row': 847046, 'end_source_row': 848845, 'start_time': '2019-08-13 20:29:43', 'duration_seconds': 1800, 'accepted_minutes_in_clip': 30, 'sha256': '1a7c5b95a0720c0d6b2c859ce3b0da6beeaa1fd032b4c9d2361ec36fb1269d4e'}
<IPython.core.display.Image object>
<IPython.core.display.Image object>


## 7. Коэффициенты и направления эффектов
Коэффициенты публикуются для каждого внешнего фолда. standardized_weight — коэффициент
базовой логистической регрессии; calibrated_weight учитывает наклон Platt-калибратора.
Среднее и SD по фолдам помогают увидеть нестабильность направления. Это ассоциации
при прочих признаках модели, не физиологические причинные эффекты.


In [8]:
coefs=pd.read_csv(work/'coefficients.csv')
display(coefs)
display(coefs.groupby(['model','feature'])[['standardized_weight','calibrated_weight']].agg(['mean','std']))


             model  fold  ... calibrated_weight  intercept
0              PPG     1  ...          0.117061  -0.558564
1              PPG     1  ...         -0.023570  -0.558564
2              PPG     1  ...          0.014652  -0.558564
3              PPG     1  ...          0.110331  -0.558564
4              PPG     1  ...         -0.156779  -0.558564
..             ...   ...  ...               ...        ...
115  PPG_plus_flow     5  ...          0.486566  -1.121631
116  PPG_plus_flow     5  ...          0.467410  -1.121631
117  PPG_plus_flow     5  ...         -0.683514  -1.121631
118  PPG_plus_flow     5  ...         -0.329122  -1.121631
119  PPG_plus_flow     5  ...          0.354433  -1.121631

[120 rows x 6 columns]
                                 standardized_weight  ... calibrated_weight
                                                mean  ...               std
model         feature                                 ...                  
PPG           PPG_SD                    

## 8. Ограничения для текста статьи
Это исследовательская классификация минут APNEA против чистого NONE на отобранных по QC окнах.
Гипопноэ исключено; AASM-скоринг, AHI и клиническая диагностика не реализованы.
PSG_Flow здесь — носовой поток/давление. Правило/модель по нему не эквивалентны AASM:
для стандартного скоринга апноэ обычно используют оронозальный термический датчик;
для гипопноэ нужны дополнительные критерии десатурации и/или араузала по выбранным правилам.
В наборе нет всех необходимых каналов/процедур для независимого полного перескоринга.

Врач исходно использовал PSG, включая поток. Это не утечка event в модель,
но flow-only получает физиологический канал, близкий к основе эталона; это другая
постановка, чем детекция только по ФПГ. Для обструктивного/центрального подтипа нужны усилия дыхания.
Нет гарантии, что добавление ФПГ улучшит flow-only: результаты сообщаются как получены.

Данные — пациенты инсультного отделения с коморбидностями. Нужны внешняя валидация,
анализ пропусков/артефактов, ручная проверка пиков и отдельная оценка всех исключённых случаев.
Для повторного выделения признаков используется raw_windows.py и встроенные patient_N.npz. Большой исходный Pickle НЕ требуется. Весь анализ и все приведённые метрики относятся только к выбранным 30 минутам на пациента, не к полным ночам. 30 пациентов и редкие события всё ещё дают широкие CI; требуется внешняя валидация.
